Intengrantes:

Sebastian Salgado

Agustin Fernandez

Eduardo Uribe

# Actividad 2.3: Pooling, ReLU, Jerarquía de Rasgos y Diseño de una CNN

Objetivo: Diseñar y analizar una CNN básica utilizando capas convolucionales, pooling y densas.

In [ ]:
import tensorflow as tf
import numpy as np

## Parte 1: Analizando una CNN existente

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3), name="entrada"),
    tf.keras.layers.Conv2D(16,(3,3),activation='relu', name="Convolucion_1"),
    tf.keras.layers.MaxPooling2D((2,2), name="Pooling_1"),
    tf.keras.layers.Conv2D(32,(3,3),activation='relu', name="Convolucion_2"),
    tf.keras.layers.MaxPooling2D((2,2), name="Pooling_2"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64,activation='relu', name="Densa_1"),
    tf.keras.layers.Dense(10,activation='softmax', name="salida")
], name="Mi_CNN")
modelo.summary()

Model: "Mi_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Convolucion_1 (Conv2D)          │ (None, 30, 30, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Pooling_1 (MaxPooling2D)        │ (None, 15, 15, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Convolucion_2 (Conv2D)          │ (None, 13, 13, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Pooling_2 (MaxPooling2D)        │ (None, 6, 6, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Densa_1 (Dense)                 │ (None, 64)             │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ salida (Dense)                  │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,530 (310.66 KB)

 Trainable params: 79,530 (310.66 KB)

 Non-trainable params: 0 (0.00 B)

### Pregunta 1
1. ¿Cuántas capas convolucionales posee?\
R: dos capas convulcionales, encargadas de extraer los patrones y características visuales de las imágenes.
2. ¿Cuántas capas de pooling posee?\
R: dos capas de pooling, intercaladas después de cada convolución para reducir las dimensiones espaciales.
3. ¿Cuál es la función de Flatten?\
R: Toma los mapas de características (que vienen en un formato 3D de alto, ancho y canales) y los estira para convertirlos en un vector unidimensional. Esto es indispensable para poder pasarle los datos a las capas densas del final, ya que estas solo procesan listas planas de numeros.
4. ¿Cuál es la función de la última capa Dense?\
R: Es la capa de clasificacion final o salida. Al tener 10 neuronas con una función de activacion softmax, se encarga de entregar las probabilidades de que la imagen de entrada pertenezca a cada una de las 10 clases posibles.

## Parte 2: Explorando ReLU

In [ ]:
feature_map=np.array([[-3,2,-1],[4,-2,5],[0,-4,3]])
relu=np.maximum(0,feature_map)
print(feature_map)
print(relu)

[[-3  2 -1]
 [ 4 -2  5]
 [ 0 -4  3]]
[[0 2 0]
 [4 0 5]
 [0 0 3]]


### Pregunta 2
1. ¿Qué valores fueron modificados?\
R: Se modificaron únicamente los valores negativos, los cuales se transformaron en 0, los valores positivos y el cero se mantuvieron exactamente igual
2. ¿Qué hace ReLU?\
R: Aplica la función matemática f(x) = \max(0, x), este actua como un filtro que apaga solamente las activaciones negativas convirtiendolas en cero, dejando pasar únicamente los valores positivos sin alterarlos
3. ¿Por qué se utiliza frecuentemente en CNN?\
R: Porque introduce no linealidad en la red con un costo computacional mínimo, ademas, mitiga el problema del desvanecimiento del gradiente


## Parte 3: Comparando MaxPooling y AveragePooling

In [ ]:
feature_map=tf.constant([[[[1.],[3.],[2.],[4.]],[[5.],[6.],[1.],[2.]],[[3.],[2.],[8.],[7.]],[[1.],[4.],[6.],[5.]]]])
print("-- MaxPooling2D --")
print(tf.keras.layers.MaxPooling2D((2,2))(feature_map).numpy()[0,:,:,0])
print("-- AveragePooling2D --")
print(tf.keras.layers.AveragePooling2D((2,2))(feature_map).numpy()[0,:,:,0])

-- MaxPooling2D --
[[6. 4.]
 [4. 8.]]
-- AveragePooling2D --
[[3.75 2.25]
 [2.5  6.5 ]]


### Pregunta 3
1. ¿Cuál conserva los valores máximos?

R: MaxPooling se queda con el número más grande de cada grupo de números, divide el mapa en grupos pequeños, y de cada grupo elige el número más grande. Esto permite conservar los rasgos visuales más prominentes y marcados (como bordes o texturas intensas) mientras descarta el ruido secundario.

2. ¿Cuál genera una representación más suavizada?

R: AveragePooling saca el promedio de los números, por eso los valores quedan más parecidos y la imagen se ve más suave, también divide el mapa en grupos de 2×2.

3. ¿Qué ventaja tiene reducir el tamaño de los mapas de características?

R: Ofrece tres ventajas clave:
1. Reducción del costo computacional: Disminuye drásticamente el volumen de datos que deben procesar las siguientes capas, optimizando el uso de memoria y acelerando el entrenamiento.
2. Control del sobreajuste:Al reducir las dimensiones espaciales, se disminuye la cantidad total de parámetros que la red debe ajustar.
3. Invarianza espacial: Le permite a la red reconocer un patrón o característica visual aunque este se haya desplazado ligeramente de posición en la imagen.



## Parte 4: Diseñando una CNN

In [ ]:
# Diseñar un modelo de red neuronal convolucional (CNN) con las siguientes capas:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3), name="entrada"),
    # Capa convolucional
    tf.keras.layers.Conv2D(16,(3,3),activation='relu', name="Convolucion_1"),
    # Agregar pooling
    tf.keras.layers.MaxPooling2D((2,2), name="Pooling_1"),
    # Capa convolucional
    tf.keras.layers.Conv2D(32,(3,3),activation='relu', name="Convolucion_2"),
    # Agregar pooling
    tf.keras.layers.MaxPooling2D((2,2), name="Pooling_2"),
    # Capa convolucional
    tf.keras.layers.Conv2D(64,(3,3),activation='relu', name="Convolucion_3"),
    # Agregar Flatten
    tf.keras.layers.Flatten(),
    # Agregar Dense oculta
    tf.keras.layers.Dense(64,activation='relu', name="Densa_1"),
    # Agregar Dense salida
    tf.keras.layers.Dense(10,activation='softmax', name="salida")
],name="NUEVO_CNN")
model.summary()



Model: "NUEVO_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Convolucion_1 (Conv2D)          │ (None, 30, 30, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Pooling_1 (MaxPooling2D)        │ (None, 15, 15, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Convolucion_2 (Conv2D)          │ (None, 13, 13, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Pooling_2 (MaxPooling2D)        │ (None, 6, 6, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Convolucion_3 (Conv2D)          │ (None, 4, 4, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Densa_1 (Dense)                 │ (None, 64)             │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ salida (Dense)                  │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,834 (350.91 KB)

 Trainable params: 89,834 (350.91 KB)

 Non-trainable params: 0 (0.00 B)

### Pregunta 4
Justifique las decisiones de diseño tomadas.

    - ¿Por qué seleccionó esa cantidad de filtros en cada capa convolucional?
    - ¿Por qué la cantidad de filtros aumenta (o no aumenta) entre capas sucesivas?
    - ¿Por qué incorporó capas de pooling en la arquitectura?
    - ¿Qué función cumple la activación ReLU dentro de la red?
    - ¿Por qué es necesario utilizar una capa Flatten antes de las capas densas?
    - ¿Qué función cumplen las capas Dense en la etapa final de la CNN?
    - ¿Cómo contribuye esta arquitectura a la construcción de una jerarquía de características (bordes → texturas → partes → objetos)?


Respuestas:
1. Arrancamos con 16 para detectar detalles simples (bordes y líneas) y subimos a 32 y 64 porque a mayor profundidad la red necesita más "detectores" para reconocer patrones más complejos.


2. Para compensar la reducción de tamaño que hace el pooling. Al achicar la imagen en ancho y alto, le damos más filtros para que aprenda rasgos abstractos sin perder información útil.



3. Para achicar la imagen a la mitad. Esto reduce la memoria necesaria, evita que la red se memorice los datos (overfitting) y hace que detecte los rasgos aunque estén un poco movidos de lugar.



4. Le da no-linealidad a la red para que pueda aprender formas complejas. Además, es súper rápida de calcular porque convierte los valores negativos en cero y evita que el gradiente se desvanezca.


5. Porque las convoluciones entregan un bloque en 3D (alto, ancho, canales) y las capas densas solo entienden vectores planos de 1D. Flatten simplemente "estira" la información.



6. Se encargan de clasificar. La primera Dense razona y combina todas las características extraídas, y la capa de salida nos da el porcentaje de probabilidad para cada clase.



7. Va de lo simple a lo complejo en 4 pasos:

1. Primera capa: detecta bordes y líneas.
2. Segunda capa: junta esos bordes en texturas y mallas.
3. Tercera capa: arma partes del objeto (ojos, orejas, ruedas).
4. Capas Dense: unen esas partes para reconocer el objeto completo.

## Parte 5: Jerarquía de características

Explique cómo evolucionan las características aprendidas por la red:

##Bordes → Texturas → Partes → Objetos

Bordes (Capas iniciales):  Se encarga de la detección de patrones locales simples (líneas, esquinas y gradientes).

Texturas (Capas intermedias): Hace una agrupación de bordes en patrones repetitivos, mallas y curvas.   

Partes (Capas avanzadas): Se encarga de la combinación de texturas para formar componentes estructurales (ojos, orejas).   

Objetos (Capas finales): Es la integración de las partes en entidades globales y completas (autos, rostros).

## Parte 6: CNN versus MLP

Compare ambas arquitecturas considerando extracción de características, parámetros e información espacial.

## Parte 7: Desafío

Reemplace MaxPooling por AveragePooling y explique las diferencias observadas.

# Parte 8: Analizando la evolución de las características en una CNN

```python
tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32,32,3)),
    tf.keras.layers.Conv2D(16,(3,3),activation='relu',padding='same',name="Conv_1"),
    tf.keras.layers.MaxPooling2D((2,2), name="Polling_1"),
    tf.keras.layers.Conv2D(32,(3,3),activation='relu',padding='same',name="Conv_2"),
    tf.keras.layers.MaxPooling2D((2,2), name="Pooling_2"),
    tf.keras.layers.Conv2D(64,(3,3),activation='relu',padding='same',name="Conv_3"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64,activation='relu',name="Densa_1"),
    tf.keras.layers.Dense(10,activation='softmax', name="salida")
], name="Otra_CNN")
```


Parte A: Dimensiones espaciales

Complete la siguiente tabla indicando las dimensiones de salida de cada capa.

| Capa                | Dimensión de salida |
| ------------------- | ------------------- |
| Entrada             | 32 × 32 × 3         |
| Conv2D (16 filtros) | ?                   |
| MaxPooling2D        | ?                   |
| Conv2D (32 filtros) | ?                   |
| MaxPooling2D        | ?                   |
| Conv2D (64 filtros) | ?                   |
| Flatten             | ?                   |


Parte B: Jerarquía de características

Indique qué tipo de patrones espera que la red aprenda principalmente en cada etapa.

| Etapa                      | Tipo de características |
| -------------------------- | ----------------------- |
| Primera capa convolucional | ?                       |
| Segunda capa convolucional | ?                       |
| Tercera capa convolucional | ?                       |
| Capa densa final           | ?                       |
